In [1]:
pip install pandas==2.2.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 5.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 5.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import requests
import concurrent.futures
import warnings
import datetime

# Get the current date and time
now = datetime.datetime.now()

# Create a datetime object representing the current date and time

# Display a message indicating what is being printed
print("Start. Current date and time : " + now.strftime("%Y-%m-%d %H:%M:%S"))

# Suppress warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Function to get IP details
def get_ip_info(ip, token, count):
    url = f"https://ipinfo.io/{ip}/json?token={token}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return {
                "ip": ip,
                "count": count,
                "city": data.get("city", "N/A"),
                "region": data.get("region", "N/A"),
                "country": data.get("country", "N/A"),
                "loc": data.get("loc", "N/A"),
                "postal": data.get("postal", "N/A"),
                "timezone": data.get("timezone", "N/A"),
                "asn": data.get("asn", {}),
                "company": data.get("company", {}),
                "privacy": data.get("privacy", {}),
                "abuse": data.get("abuse", {}),
                "domains": data.get("domains", {})
            }
        else:
            return {"ip": ip, "count": count, "error": "Could not fetch data"}
    except requests.RequestException:
        return {"ip": ip, "count": count, "error": "Request failed"}

# Load IPs from file
input_file = "latest-IP-logs.txt"  # File containing one IP per line
ips = pd.read_csv(input_file, header=None, names=["IP"])

# Count occurrences and remove duplicates
ip_counts = ips["IP"].value_counts().reset_index()
ip_counts.columns = ["IP", "Count"]

# Replace with your ipinfo.io API token
token = "8b05015358568a" # "YOUR_IPINFO_API_TOKEN"

# Use multithreading to speed up requests
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    ip_details = list(executor.map(lambda row: get_ip_info(row["IP"], token, row["Count"]), ip_counts.to_dict("records")))

# Convert to DataFrame and save
ip_df = pd.DataFrame(ip_details)
ip_df.to_csv("Output-IP-details.csv", index=False)
print("IP from nginx-proxy details saved to Output-IP-details.csv")

# Display a message indicating what is being printed
print("Finish. Current date and time : " + now.strftime("%Y-%m-%d %H:%M:%S"))

Start. Current date and time : 2025-04-13 08:49:15
IP from nginx-proxy details saved to Output-IP-details.csv
Finish. Current date and time : 2025-04-13 08:49:15


In [4]:
exit()